# Continuous Path Architecture (CPA) - Validation

Training an end-to-end CPA model on a character-level Shakespeare dataset to validate the Velocity Flow Matching objective.


## 1. Environment & Imports


In [1]:
!uv pip install -q jax jaxlib equinox optax requests


In [ ]:
import os
import requests
import jax
import jax.numpy as jnp
from jax import lax
import equinox as eqx
import optax
import time
import functools


## 2. Dataset & Tokenizer

We download `tiny-shakespeare.txt` and create a character-level tokenizer. We also define a dataloader that creates prefix paths for Strategy A.


In [149]:
import jax.numpy as jnp # Kept for the jnp.array casting below

# 1. Load dataset directly from Kaggle's local file system
text_file = "/kaggle/input/datasets/aniruddhavarma/shakespeare-text/shakespeare_copy.txt"

with open(text_file, "r", encoding="utf-8") as f:
    text = f.read()

# 2. Character-level tokenizer
chars = sorted(list(set(text)))
vocab_size = len(chars)
char2int = {c: i for i, c in enumerate(chars)}
int2char = {i: c for i, c in enumerate(chars)}

encode = lambda s: [char2int[c] for c in s]
decode = lambda l: ''.join([int2char[i] for i in l])

print(f"Dataset length: {len(text)} chars")
print(f"Vocabulary size: {vocab_size}")

# 3. Encode and cast to JAX array
data = jnp.array(encode(text), dtype=jnp.int32)

# 4. Training split (90% train, 10% validation)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

Dataset length: 5436475 chars
Vocabulary size: 84


In [150]:
# Strategy A Dataloader: Prefix Batching
# For a sequence of N tokens, we predict the next token. 
# We sample a batch of prefixes of random lengths.

def get_batch(split, batch_size, min_len=4, max_len=64, key=None):
    dataset = train_data if split == 'train' else val_data
    
    # We sample a block of size max_len + 1
    ix = jax.random.randint(key, (batch_size,), 0, len(dataset) - max_len - 1)
    
    # We also sample a random prefix length for each item in the batch
    k1, key = jax.random.split(key)
    prefix_lens = jax.random.randint(k1, (batch_size,), min_len, max_len)
    
    x = jnp.zeros((batch_size, max_len), dtype=jnp.int32)
    y = jnp.zeros((batch_size,), dtype=jnp.int32)
    mask = jnp.zeros((batch_size, max_len), dtype=jnp.bool_)
    
    # In pure JAX this loop would be vectorized, but for simplicity in dataloading 
    # we can use a python loop/numpy if done outside JIT, or vmap.
    # Let's use list comprehensions to build it and jnp.array
    # (Since this is data loading, CPU is fine)
    import numpy as np
    x_np = np.zeros((batch_size, max_len), dtype=np.int32)
    y_np = np.zeros((batch_size,), dtype=np.int32)
    mask_np = np.zeros((batch_size, max_len), dtype=np.bool_)
    
    ix_np = np.array(ix)
    prefix_lens_np = np.array(prefix_lens)
    dataset_np = np.array(dataset)
    
    for i in range(batch_size):
        idx = ix_np[i]
        plen = prefix_lens_np[i]
        
        # S[:plen]
        x_np[i, :plen] = dataset_np[idx : idx + plen]
        mask_np[i, :plen] = True
        
        # Target is the next token
        y_np[i] = dataset_np[idx + plen]
        
    return jnp.array(x_np), jnp.array(y_np), jnp.array(mask_np)


## 3. CPA Primitives

These are the mathematically rigorous Spline and CDF operations for Continuous Paths.


In [16]:
def compute_anchors(velocities: jnp.ndarray, mask: jnp.ndarray = None) -> jnp.ndarray:
    if mask is not None:
        velocities = velocities * mask[..., None]
    origin = jnp.zeros_like(velocities[:, :1, :])
    return jnp.concatenate([origin, jnp.cumsum(velocities, axis=1)], axis=1)

def _catmull_rom_tangents(anchors: jnp.ndarray) -> jnp.ndarray:
    tangents = jnp.zeros_like(anchors)
    tangents = tangents.at[:, 1:-1].set((anchors[:, 2:] - anchors[:, :-2]) / 2.0)
    tangents = tangents.at[:, 0].set(anchors[:, 1] - anchors[:, 0])
    tangents = tangents.at[:, -1].set(anchors[:, -1] - anchors[:, -2])
    return tangents

def _hermite_basis(t: jnp.ndarray):
    t2 = t * t
    t3 = t2 * t
    return (2 * t3 - 3 * t2 + 1, t3 - 2 * t2 + t, -2 * t3 + 3 * t2, t3 - t2)

def evaluate_hermite(anchors, tangents, t: jnp.ndarray, mask: jnp.ndarray) -> jnp.ndarray:
    _B, K, _d = anchors.shape
    # Instead of scaling by the static max_len, scale by the actual sequence length
    plen = jnp.sum(mask, axis=-1) # shape: (B,)
    t_scaled = t[None, :] * plen[:, None] # shape: (B, M)
    
    seg = jnp.floor(t_scaled).astype(jnp.int32).clip(0, anchors.shape[1] - 2)
    u = t_scaled - seg.astype(t.dtype)

    p0 = anchors[jnp.arange(_B)[:, None], seg]
    p1 = anchors[jnp.arange(_B)[:, None], seg + 1]
    m0 = tangents[jnp.arange(_B)[:, None], seg]
    m1 = tangents[jnp.arange(_B)[:, None], seg + 1]

    h00, h10, h01, h11 = _hermite_basis(u)
    h00, h10, h01, h11 = (x[:, :, None] for x in (h00, h10, h01, h11))
    return h00 * p0 + h10 * m0 + h01 * p1 + h11 * m1

def _catmull_rom_tangents_masked(anchors: jnp.ndarray, mask: jnp.ndarray) -> jnp.ndarray:
    tangents = jnp.zeros_like(anchors)
    tangents = tangents.at[:, 1:-1].set((anchors[:, 2:] - anchors[:, :-2]) / 2.0)
    tangents = tangents.at[:, 0].set(anchors[:, 1] - anchors[:, 0])
    
    # Fix boundary: tangent at the last valid anchor should be the forward difference of the last step
    plen = jnp.sum(mask, axis=-1)
    _B = anchors.shape[0]
    b_idx = jnp.arange(_B)
    
    # default end tangent
    tangents = tangents.at[:, -1].set(anchors[:, -1] - anchors[:, -2])
    
    # overwrite the tangent at plen with the forward difference to prevent curving back to zero
    last_tangent = anchors[b_idx, plen] - anchors[b_idx, plen - 1]
    tangents = tangents.at[b_idx, plen].set(last_tangent)
    
    return tangents

def initialize_path_hermite(velocities, M, mask=None):
    anchors = compute_anchors(velocities, mask)
    if mask is None:
        mask = jnp.ones(velocities.shape[:-1], dtype=jnp.bool_)
        
    tangents = _catmull_rom_tangents_masked(anchors, mask)
    t = jnp.linspace(0.0, 1.0, M)
    return evaluate_hermite(anchors, tangents, t, mask)


In [17]:
def speed_density(gamma: jnp.ndarray, w: jnp.ndarray, b: jnp.ndarray) -> jnp.ndarray:
    logits = jnp.einsum("bmd,d->bm", gamma, w) + b
    return jax.nn.softplus(logits) + 0.05

def build_cdf(rho: jnp.ndarray) -> jnp.ndarray:
    B, M = rho.shape
    dt = 1.0 / (M - 1)
    pieces = (rho[:, :-1] + rho[:, 1:]) / 2.0 * dt
    cdf_tail = jnp.cumsum(pieces, axis=1)
    cdf = jnp.concatenate([jnp.zeros((B, 1), dtype=rho.dtype), cdf_tail], axis=1)
    
    total = cdf[:, -1:]
    cdf = cdf / jnp.maximum(total, 1e-8)
    cdf = cdf.at[:, 0].set(0.0)
    cdf = cdf.at[:, -1].set(1.0)
    return cdf

def _resample_single(cdf: jnp.ndarray, path: jnp.ndarray, target: jnp.ndarray) -> jnp.ndarray:
    t_orig = jnp.linspace(0.0, 1.0, cdf.shape[0])
    new_t = jnp.interp(target, cdf, t_orig)
    resampled = jax.vmap(lambda col: jnp.interp(new_t, t_orig, col), in_axes=1, out_axes=1)(path)
    return resampled

def reparametrize_path(gamma: jnp.ndarray, w: jnp.ndarray, b: jnp.ndarray):
    rho = speed_density(gamma, w, b)
    cdf = build_cdf(rho)
    target = jnp.linspace(0.0, 1.0, gamma.shape[1])
    gamma_out = jax.vmap(functools.partial(_resample_single, target=target))(cdf, gamma)
    return gamma_out


## 4. CPA Network (Equinox)

We build the Deep Path Processing layer: Path-Integral Attention + MLP + Time-Warp.


In [18]:
class PathIntegralAttention(eqx.Module):
    w_q: jnp.ndarray
    w_k: jnp.ndarray
    w_v1: jnp.ndarray
    w_v2: jnp.ndarray
    alpha: float
    log_beta: jnp.ndarray

    def __init__(self, d, key):
        k1, k2, k3, k4 = jax.random.split(key, 4)
        self.w_q = jax.random.normal(k1, (d, d)) / jnp.sqrt(d)
        self.w_k = jax.random.normal(k2, (d, d)) / jnp.sqrt(d)
        self.w_v1 = jax.random.normal(k3, (d, d)) * 0.02
        self.w_v2 = jax.random.normal(k4, (d, d)) * 0.02
        self.alpha = 1.0 / jnp.sqrt(d)
        self.log_beta = jnp.array(0.54) # softplus(0.54) = 1.0, restoring strong spatial prior

    def __call__(self, gamma):
        # gamma: (B, M, d)
        M = gamma.shape[1]
        dt = 1.0 / (M - 1)
        
        # Approximate tangent velocity gamma_dot (true derivative)
        gamma_dot = jnp.zeros_like(gamma)
        gamma_dot = gamma_dot.at[:, 1:-1].set((gamma[:, 2:] - gamma[:, :-2]) / (2.0 * dt))
        gamma_dot = gamma_dot.at[:, 0].set((gamma[:, 1] - gamma[:, 0]) / dt)
        gamma_dot = gamma_dot.at[:, -1].set((gamma[:, -1] - gamma[:, -2]) / dt)

        # Value functional V(s)
        V = gamma @ self.w_v1 + gamma_dot @ self.w_v2
        
        # Kernel: Velocity Alignment + Spatial Distance
        gamma_dot_norm = gamma_dot / jnp.sqrt(jnp.mean(gamma_dot**2, axis=-1, keepdims=True) + 1e-8)
        Q = gamma_dot_norm @ self.w_q
        K = gamma_dot_norm @ self.w_k
        vel_align = jnp.einsum("btd,bsd->bts", Q, K) * self.alpha
        
        # Spatial distance ||gamma(t) - gamma(s)||^2
        dist = jnp.sum((gamma[:, :, None, :] - gamma[:, None, :, :])**2, axis=-1)
        
        logits = vel_align - jax.nn.softplus(self.log_beta) * dist
        
        # Path integral via softmax
        weights = jax.nn.softmax(logits, axis=-1)
        gamma_attn = gamma + 0.1 * jnp.einsum("bts,bsd->btd", weights, V)
        
        return gamma_attn

class ContinuousPathLayer(eqx.Module):
    attention: PathIntegralAttention
    mlp_in: eqx.nn.Linear
    mlp_out: eqx.nn.Linear
    layer_norm: eqx.nn.LayerNorm
    reparam_w: jnp.ndarray
    reparam_b: jnp.ndarray

    def __init__(self, d, key):
        k1, k2, k3, k4 = jax.random.split(key, 4)
        self.attention = PathIntegralAttention(d, k1)
        self.mlp_in = eqx.nn.Linear(d, 4 * d, key=k2)
        self.mlp_out = eqx.nn.Linear(4 * d, d, key=k3)
        self.layer_norm = eqx.nn.LayerNorm(d)
        self.reparam_w = jax.random.normal(k4, (d,)) * 0.02
        self.reparam_b = jnp.array(0.1)

    def __call__(self, gamma):
        # 1. Path-Integral Attention
        gamma = self.attention(gamma)
        
        # 2. Space Deformation (MLP) applied point-wise
        mlp_h = jax.vmap(jax.vmap(self.mlp_in))(gamma)
        mlp_h = jax.nn.gelu(mlp_h)
        gamma_deform = gamma + jax.vmap(jax.vmap(self.mlp_out))(mlp_h)
        gamma_deform = jax.vmap(jax.vmap(self.layer_norm))(gamma_deform)
        
        # 3. Reparametrization (Time Warp)
        gamma_out = reparametrize_path(gamma_deform, self.reparam_w, self.reparam_b)
        
        return gamma_out

class CPAModel(eqx.Module):
    vocab_velocities: jnp.ndarray
    layers: list
    decode_mlp1: eqx.nn.Linear
    decode_mlp2: eqx.nn.Linear
    d: int
    M: int

    def __init__(self, vocab_size, d, M, L, key):
        keys = jax.random.split(key, 3 + L)
        self.d = d
        self.M = M
        
        # Tokens are strictly their inherent semantic velocity
        self.vocab_velocities = jax.random.normal(keys[0], (vocab_size, d)) * 0.1
        
        
        self.layers = [ContinuousPathLayer(d, keys[i+2]) for i in range(L)]
        
        # Decode looks at end of path: gamma(1) and gamma_dot(1) -> 2*d
        k_dec1, k_dec2 = jax.random.split(keys[-1])
        self.decode_mlp1 = eqx.nn.Linear(2 * d, d, key=k_dec1)
        self.decode_mlp2 = eqx.nn.Linear(d, d, key=k_dec2)

    def get_normalized_vocab(self):
        return self.vocab_velocities

    def __call__(self, tokens, mask):
        # Tokens: (B, N) int array. 
        # Extract velocities
        v = self.get_normalized_vocab()[tokens] # (B, N, d)
        
        # Phase 1: Spline Init
        gamma = initialize_path_hermite(v, self.M, mask)
        
        # Phase 2: Deep Path Processing
        for layer in self.layers:
            gamma = layer(gamma)
            
        # Phase 3: Decoding (Look at endpoint t=1)
        gamma_end = gamma[:, -1, :]
        dt = 1.0 / (self.M - 1)
        gamma_dot_end = (gamma[:, -1, :] - gamma[:, -2, :]) / dt # true tangent at end
        
        decode_in = jnp.concatenate([gamma_end, gamma_dot_end], axis=-1)
        h = jax.nn.gelu(jax.vmap(self.decode_mlp1)(decode_in))
        v_pred = jax.vmap(self.decode_mlp2)(h)
        
        return v_pred

## 5. Flow Matching Loss & Training Loop


In [19]:
# Hyperparameters
d = 64
M = 32
L = 4
batch_size = 64
learning_rate = 1e-3
epochs = 50000

key = jax.random.PRNGKey(42)
k1, k2 = jax.random.split(key)

model = CPAModel(vocab_size=vocab_size, d=d, M=M, L=L, key=k1)
optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adamw(learning_rate)
)
opt_state = optimizer.init(eqx.filter(model, eqx.is_array))
@eqx.filter_jit
def loss_fn(model, x, y, mask):
    v_pred = model(x, mask) # (B, d)
    logits = v_pred @ model.get_normalized_vocab().T
    loss = optax.softmax_cross_entropy_with_integer_labels(logits, y)
    return jnp.mean(loss)
def compute_diagnostics(model, grads, x, y, mask):
    # 1. Embedding Health (Fatal)
    V = model.get_normalized_vocab()
    norm_V = V / jnp.maximum(jnp.linalg.norm(V, axis=-1, keepdims=True), 1e-8)
    cosine_sim = norm_V @ norm_V.T
    mean_cosine = (jnp.sum(cosine_sim) - V.shape[0]) / (V.shape[0] * (V.shape[0] - 1))
    
    S = jnp.linalg.svd(V, compute_uv=False)
    S_norm = S / jnp.sum(S)
    effective_rank = jnp.exp(-jnp.sum(S_norm * jnp.log(jnp.maximum(S_norm, 1e-8))))
    
    dists = jnp.sum((V[:, None, :] - V[None, :, :])**2, axis=-1)
    dists = dists + jnp.eye(V.shape[0]) * 1e6
    mean_nn_dist = jnp.mean(jnp.min(dists, axis=-1))
    
    # 2. Gradient Health (Fatal)
    reparam_grad_norm = jnp.linalg.norm(grads.layers[0].reparam_w)
    emb_grad_norm = jnp.linalg.norm(grads.vocab_velocities)
    
    total_grad_sq = sum(jnp.sum(g**2) for g in jax.tree_util.tree_leaves(grads) if g is not None)
    net_grad_norm = jnp.sqrt(jnp.maximum(total_grad_sq - emb_grad_norm**2, 1e-8))
    emb_grad_ratio = emb_grad_norm / jnp.maximum(net_grad_norm, 1e-8)
    
    # 3. Path & Attention Diagnostics (Severe)
    v = model.get_normalized_vocab()[x]
    gamma = initialize_path_hermite(v, model.M, mask)
    path_energy_in = jnp.mean(jnp.sum(gamma**2, axis=-1))
    
    attn = model.layers[0].attention
    M = gamma.shape[1]
    dt = 1.0 / (M - 1)
    gamma_dot = jnp.zeros_like(gamma)
    gamma_dot = gamma_dot.at[:, 1:-1].set((gamma[:, 2:] - gamma[:, :-2]) / (2.0 * dt))
    gamma_dot = gamma_dot.at[:, 0].set((gamma[:, 1] - gamma[:, 0]) / dt)
    gamma_dot = gamma_dot.at[:, -1].set((gamma[:, -1] - gamma[:, -2]) / dt)
    
    gamma_dot_norm = gamma_dot / jnp.sqrt(jnp.mean(gamma_dot**2, axis=-1, keepdims=True) + 1e-8)
    Q = gamma_dot_norm @ attn.w_q
    K = gamma_dot_norm @ attn.w_k
    vel_align = jnp.einsum('btd,bsd->bts', Q, K) * attn.alpha
    dist = jnp.sum((gamma[:, :, None, :] - gamma[:, None, :, :])**2, axis=-1)
    logits = vel_align - jax.nn.softplus(attn.log_beta) * dist
    weights = jax.nn.softmax(logits, axis=-1)
    
    entropy = -jnp.sum(weights * jnp.log(jnp.maximum(weights, 1e-8)), axis=-1)
    mean_attn_entropy = jnp.mean(entropy)
    dom_ratio = jnp.mean(jnp.abs(vel_align)) / jnp.maximum(jnp.mean(dist), 1e-8)
    
    for l in model.layers:
        gamma = l(gamma)
    path_energy_out = jnp.mean(jnp.sum(gamma**2, axis=-1))
    
    return {
        'eff_rank': effective_rank,
        'mean_cos': mean_cosine,
        'mean_nn_dist': mean_nn_dist,
        'reparam_grad': reparam_grad_norm,
        'emb_grad_ratio': emb_grad_ratio,
        'attn_entropy_l0': mean_attn_entropy,
        'attn_dom_ratio_l0': dom_ratio,
        'path_energy_in': path_energy_in,
        'path_energy_out': path_energy_out
    }

@eqx.filter_jit
def step(model, opt_state, x, y, mask):
    loss, grads = eqx.filter_value_and_grad(loss_fn)(model, x, y, mask)
    metrics = compute_diagnostics(model, grads, x, y, mask)
    updates, opt_state = optimizer.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss, metrics



## 6. Autoregressive Generation (Pre-Training)

Let's see what the model generates before any training.


In [20]:
def generate(model, prompt, max_new_tokens=50):
    tokens = encode(prompt)
    
    for _ in range(max_new_tokens):
        # We process the prefix
        x = jnp.array([tokens], dtype=jnp.int32)
        mask = jnp.ones_like(x, dtype=jnp.bool_)
        
        # Predict next velocity
        v_pred = model(x, mask)[0] # (d,)
        
        # Geometric Vocabulary Matching (L2 distance)
        dists = jnp.sum((model.get_normalized_vocab() - v_pred)**2, axis=-1)
        next_token = jnp.argmin(dists).item()
        
        tokens.append(next_token)
        
    return decode(tokens)

print("Generation Before Training (Random):")
out = generate(model, "ROMEO ")
print(out)


Generation Before Training (Random):
ROMEO p3MqS'_E;3MgS'_E;I80;3nM.uz((r0nM[E7z(((r0;3HNIjNj


## 7. Training Loop


In [21]:
print("Starting training...")
start_time = time.time()
key = k2

for epoch in range(epochs):
    key, subkey = jax.random.split(key)
    x, y, mask = get_batch('train', batch_size, key=subkey)
    
    model, opt_state, loss, metrics = step(model, opt_state, x, y, mask)
    
    if epoch % 50 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch:4d} | Loss: {loss:.4f} | Time: {time.time() - start_time:.1f}s")
        print(f"  [Fatal]  EffRank: {metrics['eff_rank']:.1f} | MeanCos: {metrics['mean_cos']:.3f} | NNDist: {metrics['mean_nn_dist']:.3f}")
        print(f"  [Fatal]  ReparamGrad: {metrics['reparam_grad']:.4f} | EmbGradRatio: {metrics['emb_grad_ratio']:.2f}")
        print(f"  [Severe] AttnEntropy(L0): {metrics['attn_entropy_l0']:.2f} | DomRatio(L0): {metrics['attn_dom_ratio_l0']:.2f}")
        print(f"  [Severe] PathEnergy: {metrics['path_energy_in']:.2f} -> {metrics['path_energy_out']:.2f}\n")


Starting training...
Epoch    0 | Loss: 4.5404 | Time: 17.9s
  [Fatal]  EffRank: 55.4 | MeanCos: -0.004 | NNDist: 0.812
  [Fatal]  ReparamGrad: 0.1023 | EmbGradRatio: 0.51
  [Severe] AttnEntropy(L0): 1.75 | DomRatio(L0): 0.03
  [Severe] PathEnergy: 36.34 -> 63.45

Epoch   50 | Loss: 3.2243 | Time: 36.1s
  [Fatal]  EffRank: 55.2 | MeanCos: 0.018 | NNDist: 0.828
  [Fatal]  ReparamGrad: 0.1857 | EmbGradRatio: 0.80
  [Severe] AttnEntropy(L0): 1.62 | DomRatio(L0): 0.03
  [Severe] PathEnergy: 63.37 -> 67.03

Epoch  100 | Loss: 2.5008 | Time: 37.2s
  [Fatal]  EffRank: 55.0 | MeanCos: 0.025 | NNDist: 0.837
  [Fatal]  ReparamGrad: 0.1711 | EmbGradRatio: 0.76
  [Severe] AttnEntropy(L0): 1.74 | DomRatio(L0): 0.05
  [Severe] PathEnergy: 42.75 -> 67.83

Epoch  150 | Loss: 2.5614 | Time: 38.4s
  [Fatal]  EffRank: 54.9 | MeanCos: 0.029 | NNDist: 0.844
  [Fatal]  ReparamGrad: 0.2222 | EmbGradRatio: 0.85
  [Severe] AttnEntropy(L0): 1.84 | DomRatio(L0): 0.05
  [Severe] PathEnergy: 40.87 -> 68.25

Epoch 

## 8. Autoregressive Generation (Post-Training)

Now let's see what the model generates after training.


In [24]:
print("Generation After Training:")
out = generate(model, "thy ")
print(out)


Generation After Training:
thy the for there there there them the thee are the fo


## 6. Baseline Transformer Model
To check if the number of parameters is the limiting factor, let's train a standard Transformer with a similar parameter budget (d=64, L=4).

In [ ]:
class TransformerBlock(eqx.Module):
    attention: eqx.nn.MultiheadAttention
    mlp: eqx.nn.MLP
    ln1: eqx.nn.LayerNorm
    ln2: eqx.nn.LayerNorm

    def __init__(self, d, num_heads, key):
        k1, k2 = jax.random.split(key)
        self.attention = eqx.nn.MultiheadAttention(num_heads, d, key=k1)
        self.mlp = eqx.nn.MLP(d, d, 4 * d, 1, key=k2)
        self.ln1 = eqx.nn.LayerNorm(d)
        self.ln2 = eqx.nn.LayerNorm(d)

    def __call__(self, x, mask):
        # x: (seq_len, d)
        # Causal mask
        seq_len = x.shape[0]
        causal_mask = jnp.tril(jnp.ones((seq_len, seq_len), dtype=jnp.bool_))
        
        h = jax.vmap(self.ln1)(x)
        attn_out = self.attention(h, h, h, mask=causal_mask)
        x = x + attn_out
        
        h = jax.vmap(self.ln2)(x)
        mlp_out = jax.vmap(self.mlp)(h)
        x = x + mlp_out
        return x

class TransformerModel(eqx.Module):
    token_emb: eqx.nn.Embedding
    pos_emb: eqx.nn.Embedding
    layers: list
    ln_f: eqx.nn.LayerNorm
    head: eqx.nn.Linear
    d: int
    max_len: int

    def __init__(self, vocab_size, d, L, num_heads, max_len, key):
        keys = jax.random.split(key, 3 + L)
        self.d = d
        self.max_len = max_len
        self.token_emb = eqx.nn.Embedding(vocab_size, d, key=keys[0])
        self.pos_emb = eqx.nn.Embedding(max_len, d, key=keys[1])
        self.layers = [TransformerBlock(d, num_heads, keys[i+2]) for i in range(L)]
        self.ln_f = eqx.nn.LayerNorm(d)
        self.head = eqx.nn.Linear(d, vocab_size, key=keys[-1])

    def __call__(self, tokens):
        # tokens: (seq_len,)
        seq_len = tokens.shape[0]
        pos = jnp.arange(seq_len)
        
        x = jax.vmap(self.token_emb)(tokens) + jax.vmap(self.pos_emb)(pos)
        
        for layer in self.layers:
            x = layer(x, None)
            
        x = jax.vmap(self.ln_f)(x)
        logits = jax.vmap(self.head)(x)
        return logits # (seq_len, vocab_size)


In [26]:
tf_model = TransformerModel(vocab_size=vocab_size, d=d, L=L, num_heads=4, max_len=256, key=k1)

tf_optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.adamw(learning_rate)
)
tf_opt_state = tf_optimizer.init(eqx.filter(tf_model, eqx.is_array))

@eqx.filter_jit
def tf_loss_fn(model, x, y, mask):
    # x: (B, seq_len), y: (B,), mask: (B, seq_len)
    def single_example_loss(x_i, y_i, mask_i):
        seq_len = jnp.sum(mask_i)
        logits = model(x_i) # (max_len, vocab_size)
        # We want the logit at position seq_len - 1
        pred_logits = logits[seq_len - 1]
        return optax.softmax_cross_entropy_with_integer_labels(pred_logits, y_i)
        
    losses = jax.vmap(single_example_loss)(x, y, mask)
    return jnp.mean(losses)

@eqx.filter_jit
def tf_step(model, opt_state, x, y, mask):
    loss, grads = eqx.filter_value_and_grad(tf_loss_fn)(model, x, y, mask)
    updates, opt_state = tf_optimizer.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss

print("Training Baseline Transformer...")
tf_start_time = time.time()
tf_key = k2

tf_epochs = 10000
for epoch in range(tf_epochs):
    tf_key, subkey = jax.random.split(tf_key)
    x, y, mask = get_batch('train', batch_size, key=subkey)
    
    tf_model, tf_opt_state, loss = tf_step(tf_model, tf_opt_state, x, y, mask)
    
    if epoch % 500 == 0 or epoch == tf_epochs - 1:
        print(f"Epoch {epoch:4d} | Loss: {loss:.4f} | Time: {time.time() - tf_start_time:.1f}s")


Training Baseline Transformer...
Epoch    0 | Loss: 4.7553 | Time: 6.0s
Epoch  500 | Loss: 2.6610 | Time: 17.5s
Epoch 1000 | Loss: 2.6093 | Time: 28.7s
Epoch 1500 | Loss: 2.5058 | Time: 40.4s
Epoch 2000 | Loss: 2.5640 | Time: 51.9s
Epoch 2500 | Loss: 2.1553 | Time: 63.7s
Epoch 3000 | Loss: 1.8139 | Time: 75.7s
Epoch 3500 | Loss: 2.1954 | Time: 87.3s
Epoch 4000 | Loss: 1.8710 | Time: 98.7s
Epoch 4500 | Loss: 1.9402 | Time: 110.6s
Epoch 5000 | Loss: 2.1698 | Time: 121.9s
Epoch 5500 | Loss: 2.0458 | Time: 133.5s
Epoch 6000 | Loss: 2.2614 | Time: 145.3s
Epoch 6500 | Loss: 2.1512 | Time: 156.8s
Epoch 7000 | Loss: 1.9839 | Time: 168.4s
Epoch 7500 | Loss: 2.0364 | Time: 180.4s
Epoch 8000 | Loss: 1.7998 | Time: 191.9s
Epoch 8500 | Loss: 1.4870 | Time: 203.2s
Epoch 9000 | Loss: 2.0097 | Time: 215.1s
Epoch 9500 | Loss: 1.9868 | Time: 226.9s
Epoch 9999 | Loss: 2.4313 | Time: 238.5s


In [27]:
def tf_generate(model, prompt, max_new_tokens=50):
    tokens = encode(prompt)
    
    for _ in range(max_new_tokens):
        # Convert to JAX array (seq_len,)
        x = jnp.array(tokens, dtype=jnp.int32)
        
        # The transformer expects a 1D sequence for a single example
        logits = model(x) # Shape: (seq_len, vocab_size)
        
        # Get the logits for the very last token in the sequence
        next_token_logits = logits[-1]
        
        # Greedy decoding (argmax)
        next_token = jnp.argmax(next_token_logits).item()
        
        tokens.append(next_token)
        
    return decode(tokens)

print("Transformer Generation After Training:")
out = tf_generate(tf_model, "ROMEO ")
print(out)

Transformer Generation After Training:
ROMEO SARIBE OF OF OF OR OF OF OF OR OF SERSE SERONE OF 
